# Module 4 Homework (2026)

In [ ]:

from pathlib import Path

import gdown
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import precision_score
from sklearn.tree import DecisionTreeClassifier

DATA_FILE_ID = '1oQSUMCs2DyQQIh8Y62UhrT00cIsE9Sr5'
DATA_PATH = Path('stocks_df_combined_2026_09_18.parquet.brotli')
if not DATA_PATH.exists():
    gdown.download(
        f'https://drive.google.com/uc?id={DATA_FILE_ID}',
        str(DATA_PATH),
        quiet=False,
    )

df_full = pd.read_parquet(DATA_PATH)
df_full.columns = [str(column).strip() for column in df_full.columns]
df_full.head()

,Open,High,Low,Close_x,Volume,Dividends,Stock Splits,Ticker,Year,Month,...,growth_brent_oil_7d,growth_brent_oil_30d,growth_brent_oil_90d,growth_brent_oil_365d,growth_btc_usd_1d,growth_btc_usd_3d,growth_btc_usd_7d,growth_btc_usd_30d,growth_btc_usd_90d,growth_btc_usd_365d
0,0.053745,0.061649,0.053745,0.059014,1.031789e+09,0.0,0.0,MSFT,1986,1986-03-01,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0.059014,0.062176,0.059014,0.061122,3.081600e+08,0.0,0.0,MSFT,1986,1986-03-01,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0.061122,0.062703,0.061122,0.062176,1.331712e+08,0.0,0.0,MSFT,1986,1986-03-01,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,0.062176,0.062703,0.060068,0.060595,6.776640e+07,0.0,0.0,MSFT,1986,1986-03-01,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,0.060595,0.061122,0.059014,0.059541,4.789440e+07,0.0,0.0,MSFT,1986,1986-03-01,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [22]:
# Keep the last 25 years and create the shorter categorical setup from Q1.
df_full['Date'] = pd.to_datetime(df_full['Date'], errors='coerce')
df = df_full[df_full['Date'] >= '2000-01-01'].copy()

TARGET = 'is_positive_growth_30d_future'
FUTURE_RETURN = 'growth_future_30d'
CATEGORICAL = ['Month', 'Weekday', 'ticker_type']
CATEGORICAL = [column for column in CATEGORICAL if column in df.columns]

if pd.api.types.is_datetime64_any_dtype(df['Month']):
    df['Month'] = df['Month'].dt.strftime('%B')
else:
    month_number = pd.to_numeric(df['Month'], errors='coerce')
    df['Month'] = month_number.map(
        lambda value: pd.Timestamp(2000, int(value), 1).month_name()
    )
df['Weekday'] = df['Weekday'].astype('string')
df['wom'] = ((df['Date'].dt.day - 1) // 7 + 1).astype('string')
df['month_wom'] = df['Month'].astype('string') + '_w' + df['wom']
del df['wom']

# Custom indicators used by the official Module 4 notebook.
df['ln_volume'] = np.log(df['Volume'] + 1e-6)
df['div_payout'] = (df['Dividends'] > 0).astype(int)
df['stock_split'] = (df['Stock Splits'] > 0).astype(int)

# Q1 removes Ticker and month_wom from the categorical dummy set.
dummy_variables = pd.get_dummies(df[CATEGORICAL], dtype='int32')
df_model = pd.concat([df, dummy_variables], axis=1)
DUMMIES = dummy_variables.columns.tolist()

raw_columns = {
    'Open', 'High', 'Low', 'Close_x', 'Close_y', 'Adj Close',
    'Volume', 'Dividends', 'Stock Splits', 'Year', 'Date',
    'index', 'index_x', 'index_y', 'Quarter', TARGET, FUTURE_RETURN,
}
FEATURES = [
    column for column in df_model.select_dtypes(include='number').columns
    if column not in raw_columns
    and 'future' not in column
]

# Remove duplicate column labels and keep a unique row index.
df_model = df_model.loc[:, ~df_model.columns.duplicated()].reset_index(drop=True)
FEATURES = [column for column in FEATURES if column in df_model.columns]

df_model['split'] = np.select(
    [
        df_model['Date'] <= df_model['Date'].min() + (df_model['Date'].max() - df_model['Date'].min()) * 0.70,
        df_model['Date'] <= df_model['Date'].min() + (df_model['Date'].max() - df_model['Date'].min()) * 0.85,
    ],
    ['train', 'validation'],
    default='test',
)
df_model.shape, len(FEATURES), df_model['split'].value_counts()

((202292, 230),
 208,
 split
 train         136842
 test           33163
 validation     32287
 Name: count, dtype: int64)

## Helper functions

These functions clean features, fit models, create probability rules, and calculate the simulation metrics used below.

In [23]:
def clean_features(frame, feature_columns):
    values = frame[feature_columns].replace([np.inf, -np.inf], np.nan)
    return pd.DataFrame(
        SimpleImputer(strategy='constant', fill_value=0).fit_transform(values),
        index=frame.index,
        columns=feature_columns,
    )

def add_splits(frame):
    train = frame[frame['split'].isin(['train', 'validation'])]
    test = frame[frame['split'].eq('test')]
    X_train = clean_features(train, FEATURES)
    y_train = train[TARGET].astype(int)
    X_test = clean_features(test, FEATURES)
    y_test = test[TARGET].astype(int)
    X_all = clean_features(frame, FEATURES)
    return train, test, X_train, y_train, X_test, y_test, X_all

def simulate_prediction(frame, prediction_column):
    test = frame[frame['split'].eq('test')].copy()
    test['gross_revenue'] = (
        test[prediction_column] * 100 * (test[FUTURE_RETURN] - 1)
    )
    test['fees'] = -test[prediction_column] * 100 * 0.002
    net_revenue = (test['gross_revenue'] + test['fees']).sum()
    trades = int(test[prediction_column].sum())
    investments_per_day = test.loc[
        test[prediction_column].eq(1)
    ].groupby('Date')[prediction_column].count()
    avg_trades_per_day = (
        trades / test['Date'].nunique()
        if test['Date'].nunique()
        else 0
    )
    q75_trades_per_day = (
        investments_per_day.quantile(0.75)
        if not investments_per_day.empty
        else 0
    )
    capital = 100 * 30 * q75_trades_per_day
    cagr = (
        ((capital + net_revenue) / capital) ** (1 / 4)
        if capital > 0 and capital + net_revenue > 0
        else np.nan
    )
    return {
        'predictor': prediction_column,
        'trades': trades,
        'avg_trades_per_day': avg_trades_per_day,
        'q75_trades_per_day': q75_trades_per_day,
        'precision': precision_score(
            test[TARGET],
            test[prediction_column],
            zero_division=0,
        ),
        'net_revenue': net_revenue,
        'capital': capital,
        'CAGR': cagr,
    }

## Question 1: Less features is more

Use the shorter dummy-variable setup above, fit the best Random Forest from the lecture (`max_depth=13`, `n_estimators=100`), create the 0.55 probability rule, and calculate its CAGR.

In [24]:
train_df, test_df, X_train, y_train, X_test, y_test, X_all = add_splits(df_model)
rf_best = RandomForestClassifier(
    max_depth=13,
    n_estimators=100,
    random_state=42,
    n_jobs=-1,
)
rf_best.fit(X_train, y_train)

y_pred_all_class1_array = rf_best.predict_proba(X_all)[:, 1]
df_model['pred10_rf_best_rule_55'] = (y_pred_all_class1_array >= 0.55).astype(int)
q1_result = simulate_prediction(df_model, 'pred10_rf_best_rule_55')
q1_result

{'predictor': 'pred10_rf_best_rule_55',
 'trades': 5554,
 'avg_trades_per_day': 5.3403846153846155,
 'q75_trades_per_day': np.float64(11.25),
 'precision': 0.6127115592365863,
 'net_revenue': np.float64(18067.154003156833),
 'capital': np.float64(33750.0),
 'CAGR': np.float64(1.1131403988665511)}

## Question 2: Most efficient Random Forest decision rule

Create probability rules from 0.55 through 0.65. Compare precision, CAGR, number of trades, and average trades per day. Select the best CAGR among rules with average trades per day below 5.

In [26]:
# Add the existing 0.55 rule and the new thresholds from 0.56 to 0.65.
thresholds = np.round(np.arange(0.55, 0.66, 0.01), 2)
rule_columns = []
for number, threshold in enumerate(thresholds, start=10):
    column = f'pred{number}_rf_best_rule_{int(threshold * 100):02d}'
    df_model[column] = (y_pred_all_class1_array >= threshold).astype(int)
    rule_columns.append(column)

simulation_results = pd.DataFrame([
    simulate_prediction(df_model, column)
    for column in rule_columns
])
simulation_results = simulation_results.sort_values(
    'CAGR',
    ascending=False,
).reset_index(drop=True)

efficient_rules = simulation_results[
    simulation_results['avg_trades_per_day'] < 5
].copy()
if efficient_rules.empty:
    raise ValueError('No Random Forest rule has fewer than 5 average trades per day.')

second_best_rule = efficient_rules.iloc[0]
print('Second-best rule below 5 average trades/day:')
print(second_best_rule)
simulation_results

Second-best rule below 5 average trades/day:
predictor             pred11_rf_best_rule_56
trades                                  4201
avg_trades_per_day                  4.039423
q75_trades_per_day                      10.0
precision                           0.614139
net_revenue                      14454.34667
capital                              30000.0
CAGR                                1.103312
Name: 1, dtype: object


,predictor,trades,avg_trades_per_day,q75_trades_per_day,precision,net_revenue,capital,CAGR
0,pred10_rf_best_rule_55,5554,5.340385,11.25,0.612712,18067.154003,33750.0,1.113140
1,pred11_rf_best_rule_56,4201,4.039423,10.00,0.614139,14454.346670,30000.0,1.103312
2,pred12_rf_best_rule_56,3011,2.895192,9.00,0.610096,10737.946486,27000.0,1.087311
3,pred13_rf_best_rule_57,2107,2.025962,7.00,0.616991,7823.238163,21000.0,1.082383
4,pred14_rf_best_rule_59,1417,1.362500,7.00,0.637968,6196.941211,21000.0,1.066781
5,pred15_rf_best_rule_60,937,0.900962,5.00,0.637140,4385.765050,15000.0,1.066223
6,pred16_rf_best_rule_61,577,0.554808,4.00,0.668977,3169.915140,12000.0,1.060353
7,pred17_rf_best_rule_62,346,0.332692,3.00,0.684971,2164.753101,9000.0,1.055362
8,pred18_rf_best_rule_63,202,0.194231,3.00,0.702970,1590.445754,9000.0,1.041521
9,pred19_rf_best_rule_64,116,0.111538,2.50,0.732759,1102.621326,7500.0,1.034886


## Question 3: Redefining the growth target

Start from a fresh copy. Redefine the positive target as `growth_future_30d > 1.022`, rerun the model, and report the number of test trades for `pred5_clf_10`.

In [27]:
# Run this section from a fresh copy, after rerunning the setup cells.
df_model[TARGET] = (df_model[FUTURE_RETURN] > 1.022).astype(int)
train_df, test_df, X_train, y_train, X_test, y_test, X_all = add_splits(df_model)
clf10 = DecisionTreeClassifier(max_depth=10, random_state=42)
clf10.fit(X_train, y_train)
df_model['pred5_clf_10'] = clf10.predict(X_all)
q3_trades = int(df_model.loc[df_model['split'].eq('test'), 'pred5_clf_10'].sum())
print('Q3 number of trades:', q3_trades)

Q3 number of trades: 7244


## Question 4: Describe your ideal trading strategy

A reasonable strategy would trade only the strongest model probabilities, use equal or volatility-adjusted position sizes, cap exposure per ticker, and include transaction costs. I would validate the rules on a later test period and compare CAGR, drawdown, turnover, and Sharpe ratio with a buy-and-hold benchmark.